# Filtering rows with WHERE

Practice notebook for the datafromzero SQL lesson **Filtering rows with WHERE**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Compare a number

In [ ]:
con.sql("""
SELECT product_name, unit_price_usd
FROM products
WHERE unit_price_usd >= 4.50
""").show()

### Step 2. Combine two conditions with AND

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders
FROM orders
WHERE store_id = 1 AND payment_method = 'mobile'
""").show()

### Step 3. Add OR, the wrong way

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders
FROM orders
WHERE store_id = 1 AND payment_method = 'mobile' OR payment_method = 'cash'
""").show()

### Step 4. Put parentheses around the OR

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders
FROM orders
WHERE store_id = 1 AND (payment_method = 'mobile' OR payment_method = 'cash')
""").show()

### Step 5. Shorten it with IN

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders
FROM orders
WHERE store_id = 1 AND payment_method IN ('mobile', 'cash')
""").show()

### Step 6. Add the date range

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders
FROM orders
WHERE store_id = 1
  AND payment_method IN ('mobile', 'cash')
  AND ordered_at >= '2026-03-01'
  AND ordered_at < '2026-04-01'
""").show()

### Step 7. See why not BETWEEN for dates

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders
FROM orders
WHERE ordered_at >= '2026-03-01' AND ordered_at < '2026-04-01'
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders
FROM orders
WHERE ordered_at BETWEEN '2026-03-01' AND '2026-03-31'
""").show()

### Step 8. Match text and a range

In [ ]:
con.sql("""
SELECT product_name, unit_price_usd
FROM products
WHERE product_name LIKE '%Latte%' AND unit_price_usd BETWEEN 4.00 AND 5.00
""").show()

### Step 9. Verify

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders
FROM orders
WHERE store_id = 1
  AND NOT (payment_method IN ('mobile', 'cash'))
  AND ordered_at >= '2026-03-01'
  AND ordered_at < '2026-04-01'
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders
FROM orders
WHERE store_id = 1
  AND ordered_at >= '2026-03-01'
  AND ordered_at < '2026-04-01'
""").show()

## Exercise 1. Campus card orders in April 2026

Campus is `store_id` 2. The first three lines are given and count Campus card orders of all time. Add the two lines that
keep April only: from its first day up to, but not including, May 1.

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders
FROM orders
WHERE store_id = 2 AND payment_method = 'card'
""").show()

## Exercise 2. Coffee and tea under 4.00 dollars

List the coffee and tea items that cost less than 4.00 US dollars: name, category and price, cheapest first, equal prices in
alphabetical order. Before you run anything, write one sentence in the first cell saying what the query will return.

In [ ]:
# My expected result, in words:
#

In [ ]:
# Your query: coffee and tea items under 4.00

## Exercise 3. Spot the flaw

A colleague counts the orders of June 2026 with this query and gets 344. It runs without an error, but the data holds
more June orders. Find the flaw, then write a query that counts them all.

In [ ]:
con.sql("""
SELECT COUNT(*) AS orders
FROM orders
WHERE ordered_at BETWEEN '2026-06-01' AND '2026-06-30'
""").show()

In [ ]:
# Your corrected query: